# Bell Pre-Lab

## Given data

We are given coincidence counts for a collection of two-qubit projectors of the form
$$
\Pi_\nu = |x\rangle\langle x|\otimes|y\rangle\langle y| = |\psi_\nu\rangle\langle\psi_\nu|,
$$
where $x,y\in\{H,V,D,A,R,L\}$ label single-photon polarization states.

We use these counts directly to reconstruct a physical two-qubit density matrix $\rho$.

## Q1 — Two-photon MLE tomography


For each measurement index $\nu$, we model the expected number of coincidences as
$$
\mu_\nu = N\,p_\nu(\rho),
\qquad
p_\nu(\rho)=\langle\psi_\nu|\rho|\psi_\nu\rangle=\mathrm{Tr}(\Pi_\nu\rho).
$$
Here $N$ is a single global scale factor capturing overall brightness / integration time, and $\Pi_\nu=|\psi_\nu\rangle\langle\psi_\nu|$ is the two-qubit projector associated with that count.

Because coincidences are photon-counting measurements, we assume Poisson shot noise:
$$
n_\nu \sim \mathrm{Poisson}(\mu_\nu),
\qquad
\mathrm{Var}(n_\nu)=\mu_\nu.
$$

To match the notes exactly, we estimate $\rho$ by minimizing the Poisson-variance weighted least-squares objective
$$
\mathcal{L}(\rho,N)=\sum_{\nu=1}^{k}\frac{(\mu_\nu-n_\nu)^2}{2\,\mu_\nu}
=\sum_{\nu=1}^{k}\frac{\big(Np_\nu(\rho)-n_\nu\big)^2}{2\,Np_\nu(\rho)}.
$$
This weighting is appropriate because the Poisson variance scales with the mean, so high-count measurements (larger $\mu_\nu$) are automatically down-weighted relative to low-count measurements.

Finally, we enforce physicality of $\rho$ using the Cholesky parametrization
$$
\rho(\mathbf{t})=\frac{T^\dagger(\mathbf{t})T(\mathbf{t})}{\mathrm{Tr}\!\left[T^\dagger(\mathbf{t})T(\mathbf{t})\right]},
$$
which guarantees $\rho\succeq 0$ and $\mathrm{Tr}(\rho)=1$. We fit both $\mathbf{t}$ and $N>0$ (using $N=e^\eta$ to guarantee positivity).

In [1]:
import numpy as np
from scipy.optimize import minimize

ketH = np.array([1, 0], dtype=complex)
ketV = np.array([0, 1], dtype=complex)
ketD = (ketH + ketV)/np.sqrt(2)
ketA = (ketH - ketV)/np.sqrt(2)
ketR = (ketH + 1j*ketV)/np.sqrt(2)
ketL = (ketH - 1j*ketV)/np.sqrt(2)

kets = {"H": ketH, "V": ketV, "D": ketD, "A": ketA, "R": ketR, "L": ketL}

def projector(lbl: str) -> np.ndarray:
    v = kets[lbl].reshape(2, 1)
    return v @ v.conj().T

def twoqubit_projector(outcome: str) -> np.ndarray:
    return np.kron(projector(outcome[0]), projector(outcome[1]))

data = {
    "HH": 34749, "HV": 324,  "VH": 444,  "VV": 35805,
    "HD": 17238, "HL": 16722, "DH": 16901, "RH": 16324,
    "DD": 32028, "RD": 15132, "RL": 33586, "DR": 17932,
    "DV": 13441, "RV": 17521, "VD": 13171, "VL": 17170
}

outcomes = list(data.keys())
ns = np.array([data[o] for o in outcomes], dtype=float)
Pis = np.stack([twoqubit_projector(o) for o in outcomes], axis=0)
Ntot = ns.sum()

def rho_from_params(x: np.ndarray) -> np.ndarray:
    """
    x has 16 real parameters:
      - 4 real diagonal entries of lower-triangular T
      - 6 complex lower-triangular off-diagonal entries = 12 real numbers
    """
    T = np.zeros((4, 4), dtype=complex)
    idx = 0

    for i in range(4):
        T[i, i] = x[idx]
        idx += 1

    for i in range(1, 4):
        for j in range(i):
            T[i, j] = x[idx] + 1j*x[idx + 1]
            idx += 2

    M = T.conj().T @ T
    return M / np.trace(M)

def chi2_poisson_objective(y):
    x = y[:-1]
    eta = y[-1]
    N = np.exp(eta)

    rho = rho_from_params(x)
    p = np.real(np.einsum("kij,ji->k", Pis, rho))
    p = np.clip(p, 1e-12, None)

    mu = N * p                                      
    return np.sum((mu - ns)**2 / (2.0 * mu))

def mle_tomography_chi2(n_starts=25, seed=0):
    rng = np.random.default_rng(seed)
    best = None

    for _ in range(n_starts):
        x0 = np.zeros(16)
        x0[:4] = 1.0 + 0.2*rng.standard_normal(4)
        x0[4:] = 0.2*rng.standard_normal(12)

        rho0 = rho_from_params(x0)
        ps0 = np.real(np.einsum("kij,ji->k", Pis, rho0))
        ps0 = np.clip(ps0, 1e-12, None)
        N0 = float(ns.sum() / ps0.sum())
        eta0 = np.log(max(N0, 1e-12))

        y0 = np.concatenate([x0, [eta0]])

        res = minimize(
            chi2_poisson_objective, y0,
            method="Powell",
            options=dict(maxiter=40000, xtol=1e-8, ftol=1e-8)
        )

        if best is None or res.fun < best.fun:
            best = res

    x_hat = best.x[:-1]
    eta_hat = best.x[-1]
    N_hat = float(np.exp(eta_hat))
    rho_hat = rho_from_params(x_hat)
    return rho_hat, N_hat, best

rho_chi2, N_chi2, fit_chi2 = mle_tomography_chi2()
eig_chi2 = np.linalg.eigvalsh(rho_chi2)

print("CHI2 fit success:", fit_chi2.success)
print("Objective:", fit_chi2.fun)
print("N_hat:", N_chi2)
print("Eigenvalues:", eig_chi2)
print("\nReconstructed rho (|HH>,|HV>,|VH>,|VV>):\n", rho_chi2)

CHI2 fit success: True
Objective: 343.90593594276186
N_hat: 71510.0052968602
Eigenvalues: [1.31839545e-14 4.86179617e-13 3.51053828e-02 9.64894617e-01]

Reconstructed rho (|HH>,|HV>,|VH>,|VV>):
 [[ 0.50322147+0.j         -0.02134926-0.01143952j -0.02499316+0.01894759j
   0.46618593-0.02188369j]
 [-0.02134926+0.01143952j  0.00522532+0.j          0.00408703+0.00170905j
  -0.03252171+0.005722j  ]
 [-0.02499316-0.01894759j  0.00408703-0.00170905j  0.0072378 +0.j
  -0.03966007-0.01135973j]
 [ 0.46618593+0.02188369j -0.03252171-0.005722j   -0.03966007+0.01135973j
   0.48431541+0.j        ]]


In the computational polarization basis
$$
\{|HH\rangle,\ |HV\rangle,\ |VH\rangle,\ |VV\rangle\},
$$
the reconstructed density matrix is
$$
\rho \approx
\begin{pmatrix}
0.503221 & -0.021349-0.011440i & -0.024993+0.018948i & 0.466186-0.021884i\\
-0.021349+0.011440i & 0.005225 & 0.004087+0.001709i & -0.032522+0.005722i\\
-0.024993-0.018948i & 0.004087-0.001709i & 0.007238 & -0.039660-0.011360i\\
0.466186+0.021884i & -0.032522-0.005722i & -0.039660+0.011360i & 0.484315
\end{pmatrix}.
$$

As a physicality check, the eigenvalues are approximately
$$
\{1.32\times 10^{-14},\ 4.86\times 10^{-13},\ 0.03511,\ 0.96489\},
$$
which are non-negative (up to numerical precision) and sum to $1$. The dominant eigenvalue close to $1$ indicates a high-purity Bell-like state.

## Q2 — Fidelity with the Bell states

For a reconstructed state $\rho$ and a target pure Bell state $|\psi\rangle$, the fidelity is
$$
F(\psi)=\langle\psi|\rho|\psi\rangle.
$$

We compute this for the four Bell states
$$
|\Phi^\pm\rangle=\frac{|HH\rangle\pm|VV\rangle}{\sqrt{2}},\qquad
|\Psi^\pm\rangle=\frac{|HV\rangle\pm|VH\rangle}{\sqrt{2}}.
$$

The Bell state with the largest fidelity identifies the closest Bell component of the reconstructed state.

In [2]:
ketH = np.array([1, 0], dtype=complex)
ketV = np.array([0, 1], dtype=complex)

ket00 = np.kron(ketH, ketH)
ket01 = np.kron(ketH, ketV)
ket10 = np.kron(ketV, ketH)
ket11 = np.kron(ketV, ketV)

bell = {
    "Phi+": (ket00 + ket11)/np.sqrt(2),
    "Phi-": (ket00 - ket11)/np.sqrt(2),
    "Psi+": (ket01 + ket10)/np.sqrt(2),
    "Psi-": (ket01 - ket10)/np.sqrt(2),
}

def fidelity_pure(rho, psi):
    psi = psi.reshape(-1, 1)
    return float(np.real((psi.conj().T @ rho @ psi)[0, 0]))

bell_fids = {name: fidelity_pure(rho_chi2, psi) for name, psi in bell.items()}
bell_fids

{'Phi+': 0.9599543660208094,
 'Phi-': 0.027582515326098865,
 'Psi+': 0.010318589112030886,
 'Psi-': 0.0021445295410608372}

The Bell-state fidelities are

 $$F(\Phi^+) = 0.95995$$  
 $$F(\Phi^-) = 0.02758$$  
 $$F(\Psi^+) = 0.01032$$  
 $$F(\Psi^-) = 0.00214$$  

The largest fidelity is $F(\Phi^+)$, so the reconstructed state is closest to
$$
|\Phi^+\rangle=\frac{|HH\rangle+|VV\rangle}{\sqrt{2}}.
$$

## Q3 — Correlations for arbitrary analyzer directions

A single-qubit measurement along a Bloch-sphere direction $\mathbf{n}$ corresponds to the observable
$$
\mathbf{n}\cdot\boldsymbol{\sigma}=n_x\sigma_x+n_y\sigma_y+n_z\sigma_z,
$$
where $\boldsymbol{\sigma}=(\sigma_x,\sigma_y,\sigma_z)$ and $\|\mathbf{n}\|=\|\mathbf{m}\|=1$.

For two qubits, the correlation for settings $\mathbf{n}$ (Alice) and $\mathbf{m}$ (Bob) is
$$
E(\mathbf{n},\mathbf{m})=\mathrm{Tr}\!\left[\big(\mathbf{n}\cdot\boldsymbol{\sigma}\otimes\mathbf{m}\cdot\boldsymbol{\sigma}\big)\rho\right].
$$

We parameterize unit vectors with spherical angles
$$
\mathbf{n}=(\sin\theta\cos\phi,\ \sin\theta\sin\phi,\ \cos\theta),
$$
and similarly for $\mathbf{m}$. This correlation function is used in Q4 to form the CHSH parameter.


In [3]:
sx = np.array([[0, 1],[1, 0]], dtype=complex)
sy = np.array([[0, -1j],[1j, 0]], dtype=complex)
sz = np.array([[1, 0],[0, -1]], dtype=complex)

def unitvec(theta, phi):
    return np.array([np.sin(theta)*np.cos(phi),
                     np.sin(theta)*np.sin(phi),
                     np.cos(theta)], dtype=float)

def n_dot_sigma(n):
    return n[0]*sx + n[1]*sy + n[2]*sz

def correlation(rho, n, m):
    return float(np.real(np.trace(np.kron(n_dot_sigma(n), n_dot_sigma(m)) @ rho)))

x = np.array([1.0, 0.0, 0.0])
y = np.array([0.0, 1.0, 0.0])
z = np.array([0.0, 0.0, 1.0])

Ezz = correlation(rho_chi2, z, z)
Exx = correlation(rho_chi2, x, x)
Eyy = correlation(rho_chi2, y, y)

Ezz, Exx, Eyy

(0.9750737626938166, 0.9405459102656807, -0.9241977911237406)

Using
$$
E(\mathbf{n},\mathbf{m})=\mathrm{Tr}\!\left[\big(\mathbf{n}\cdot\boldsymbol{\sigma}\otimes\mathbf{m}\cdot\boldsymbol{\sigma}\big)\rho\right],
$$
the reconstructed state exhibits strong correlations along the principal axes. In particular,

 $$E(\hat z,\hat z) \approx 0.97507,$$
 $$E(\hat x,\hat x) \approx 0.94055,$$
 $$E(\hat y,\hat y) \approx -0.92420.$$

These values lie within $[-1,1]$ as required and are used directly in the CHSH parameter in Q4.

## Q4 — CHSH parameter: compute and optimize

The CHSH parameter is defined as
$$
S = E(\mathbf{a},\mathbf{b}) + E(\mathbf{a},\mathbf{b}') + E(\mathbf{a}',\mathbf{b}) - E(\mathbf{a}',\mathbf{b}').
$$

A local hidden-variable (classical) model obeys the Bell–CHSH inequality
$$
|S| \le 2,
$$
while quantum mechanics allows up to Tsirelson’s bound
$$
|S| \le 2\sqrt{2}.
$$

We numerically maximize $|S|$ over four analyzer directions $\mathbf{a},\mathbf{a}',\mathbf{b},\mathbf{b}'$ for:
1. an ideal Bell state,
2. a separable product state $|HH\rangle$,
3. the reconstructed experimental state $\rho$ from Q1.

In [4]:
def chsh_S(rho, a, ap, b, bp):
    return (correlation(rho,a,b) + correlation(rho,a,bp)
            + correlation(rho,ap,b) - correlation(rho,ap,bp))

def optimize_chsh(rho, n_starts=150, seed=3):
    rng = np.random.default_rng(seed)
    bounds = [(0,np.pi),(0,2*np.pi)]*4  # (theta,phi) for a, a', b, b'

    def unpack(x):
        return (unitvec(x[0],x[1]),
                unitvec(x[2],x[3]),
                unitvec(x[4],x[5]),
                unitvec(x[6],x[7]))

    def obj(x):
        a, ap, b, bp = unpack(x)
        return -abs(chsh_S(rho, a, ap, b, bp))

    best = None
    for _ in range(n_starts):
        x0 = np.array([rng.uniform(lo,hi) for lo,hi in bounds])
        res = minimize(obj, x0, method="L-BFGS-B", bounds=bounds,
                       options=dict(maxiter=1200))
        if best is None or res.fun < best.fun:
            best = res

    a, ap, b, bp = unpack(best.x)
    Smax = float(abs(chsh_S(rho, a, ap, b, bp)))
    return Smax, (a, ap, b, bp)

ketH = np.array([1,0], dtype=complex)
ketV = np.array([0,1], dtype=complex)

ket00 = np.kron(ketH, ketH)
ket11 = np.kron(ketV, ketV)

phi_plus = (ket00 + ket11)/np.sqrt(2)
rho_bell = phi_plus.reshape(-1,1) @ phi_plus.conj().reshape(1,-1)

rho_HH = ket00.reshape(-1,1) @ ket00.conj().reshape(1,-1)


S_bell, _ = optimize_chsh(rho_bell, n_starts=60, seed=1)
S_HH, _   = optimize_chsh(rho_HH,   n_starts=60, seed=2)
S_rho, settings_rho = optimize_chsh(rho_chi2, n_starts=200, seed=3)

S_bell, S_HH, S_rho, settings_rho

(2.828427124746186,
 2.0,
 2.7181012334740693,
 (array([ 0.70949763, -0.02842928,  0.70413414]),
  array([ 0.59549568,  0.48305134, -0.64190833]),
  array([-0.95979053,  0.27575939, -0.05252532]),
  array([-0.09583981, -0.33735993, -0.93648439])))

From numerical optimization of
$$
S = E(\mathbf{a},\mathbf{b}) + E(\mathbf{a},\mathbf{b}') + E(\mathbf{a}',\mathbf{b}) - E(\mathbf{a}',\mathbf{b}'),
$$
we obtain:

- Ideal Bell state: $|S|_{\max}=2.82843\approx 2\sqrt{2}.$ 
- Product state $|HH\rangle$: $|S|_{\max}=2.00000$ (no violation).  
- Reconstructed experimental state $\rho$: $|S|_{\max}=2.71810.$

Thus, the reconstructed state violates the Bell–CHSH inequality ($|S|>2$) and the violation is close to the quantum maximum $2\sqrt{2}$, consistent with a high-fidelity Bell-like state.

One optimal set of analyzer directions (unit vectors) achieving the maximum for the reconstructed state is:
$$
\mathbf{a}\approx(0.709498,\,-0.028429,\,0.704134),\qquad
\mathbf{a}'\approx(0.595496,\,0.483051,\,-0.641908),
$$
$$
\mathbf{b}\approx(-0.959791,\,0.275759,\,-0.052525),\qquad
\mathbf{b}'\approx(-0.095840,\,-0.337360,\,-0.936484).
$$

## Q5 — QWP then HWP angles for the optimal measurement settings

To measure along a Bloch direction $\mathbf{n}$, we choose a single-qubit unitary $U$ that maps the eigenbasis of $\mathbf{n}\cdot\boldsymbol{\sigma}$ to the $H/V$ basis, so that a standard PBS measurement implements the desired observable.

In the lab, the analyzer is implemented with a **quarter-wave plate (QWP)** followed by a **half-wave plate (HWP)**:
$$
U \approx U_{\mathrm{HWP}}(\beta)\,U_{\mathrm{QWP}}(\alpha),
$$
up to a global phase. For each optimal direction $\mathbf{a},\mathbf{a}',\mathbf{b},\mathbf{b}'$ from Q4, we numerically fit $(\alpha,\beta)$ and report a fit error (where $0$ indicates a perfect match).

In [5]:
def n_dot_sigma(n):
    return n[0]*sx + n[1]*sy + n[2]*sz

def jones_QWP(alpha):
    c, s = np.cos(alpha), np.sin(alpha)
    R = np.array([[c, -s],[s, c]], dtype=complex)
    J = np.array([[1, 0],[0, 1j]], dtype=complex)
    return R.conj().T @ J @ R

def jones_HWP(beta):
    c, s = np.cos(beta), np.sin(beta)
    R = np.array([[c, -s],[s, c]], dtype=complex)
    J = np.array([[1, 0],[0, -1]], dtype=complex)
    return R.conj().T @ J @ R

def su2_for_axis(n):
    A = n_dot_sigma(n)
    w, v = np.linalg.eigh(A)
    ket_np = v[:, np.argmax(w)]
    ket_nm = v[:, np.argmin(w)]
    U = np.column_stack([ket_np, ket_nm])
    U = U / np.sqrt(np.linalg.det(U))
    return U

def fit_qwp_hwp(U_target, n_starts=200, seed=10):
    rng = np.random.default_rng(seed)

    def dist(x):
        alpha, beta = x
        U = jones_HWP(beta) @ jones_QWP(alpha)  # QWP then HWP
        overlap = np.trace(U.conj().T @ U_target) / 2.0
        return 1 - np.abs(overlap)  # 0 is perfect (up to global phase)

    bounds = [(0, 2*np.pi), (0, 2*np.pi)]
    best = None
    for _ in range(n_starts):
        x0 = np.array([rng.uniform(0, 2*np.pi), rng.uniform(0, 2*np.pi)])
        res = minimize(dist, x0, method="L-BFGS-B", bounds=bounds, options=dict(maxiter=2500))
        if best is None or res.fun < best.fun:
            best = res

    alpha, beta = best.x
    return (float(np.degrees(alpha) % 180), float(np.degrees(beta) % 180), float(best.fun))

a, ap, b, bp = settings_rho

angles = {
    "a":  fit_qwp_hwp(su2_for_axis(a),  seed=11),
    "a'": fit_qwp_hwp(su2_for_axis(ap), seed=12),
    "b":  fit_qwp_hwp(su2_for_axis(b),  seed=13),
    "b'": fit_qwp_hwp(su2_for_axis(bp), seed=14),
}
angles

{'a': (22.62018747564079, 101.31009581800647, 0.2788767560300329),
 "a'": (115.03286636499692, 147.51643326028068, 0.0972340565466755),
 'b': (136.50543220962214, 23.25271636926167, 0.20120698159807782),
 "b'": (79.7346487319495, 39.86732465867013, 0.009561671062101706)}

The best-fit waveplate angles for a **QWP followed by HWP** implementing the four optimal analyzer directions are:

| Setting | QWP (°) | HWP (°) | Fit error |
|---|---:|---:|---:|
| $a$  | 22.620 | 101.310 | 0.2789 |
| $a'$ | 115.033 | 147.516 | 0.0972 |
| $b$  | 136.505 | 23.253 | 0.2012 |
| $b'$ | 79.735 | 39.867 | 0.0096 |

The fit error quantifies how closely the two-plate sequence reproduces the target single-qubit rotation (up to global phase). Smaller values indicate a more accurate realization with only a QWP followed by an HWP.